# OmniParser: run on one image
1. Run the **Setup** cell once (loads models).
2. Paste an image path into `IMAGE_PATH` in the **Run** cell and run it. Repeat as often as you like.

Outputs go to `results/<image-name>_<timestamp>/`: the labelled image (`labelled.png`) and the parsed elements (`elements.csv`).
Use the `omni` kernel/env.

In [ ]:
# Setup: run once
import os, io, time, base64
from datetime import datetime
from pathlib import Path
import torch, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

REPO = Path('/mnt/hitchcock/scratch/oberon/OmniParser')
os.chdir(REPO)
from util.utils import get_som_labeled_img, check_ocr_box, get_caption_model_processor, get_yolo_model

DEVICE = torch.device('cuda:0')
RESULTS_DIR = REPO / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

som_model = get_yolo_model('weights/icon_detect/model.pt').to(DEVICE)
caption_model_processor = get_caption_model_processor(
    model_name='florence2', model_name_or_path='weights/icon_caption_florence', device=DEVICE)
print('models loaded on', DEVICE)

In [ ]:
def run_omniparser(image_path, box_threshold=0.05, iou_threshold=0.7, show=True):
    image_path = Path(str(image_path).strip().strip('"\''))
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    image = Image.open(image_path).convert('RGB')

    ratio = max(image.size) / 3200
    draw_bbox_config = {
        'text_scale': 0.8 * ratio,
        'text_thickness': max(int(2 * ratio), 1),
        'text_padding': max(int(3 * ratio), 1),
        'thickness': max(int(3 * ratio), 1),
    }

    t0 = time.time()
    (ocr_text, ocr_bbox), _ = check_ocr_box(
        image, display_img=False, output_bb_format='xyxy', goal_filtering=None,
        easyocr_args={'paragraph': False, 'text_threshold': 0.9}, use_paddleocr=True)
    labelled_b64, _, parsed = get_som_labeled_img(
        image, som_model, BOX_TRESHOLD=box_threshold, output_coord_in_ratio=True,
        ocr_bbox=ocr_bbox, draw_bbox_config=draw_bbox_config,
        caption_model_processor=caption_model_processor, ocr_text=ocr_text,
        use_local_semantics=True, iou_threshold=iou_threshold, scale_img=False, batch_size=128)
    elapsed = time.time() - t0

    out_dir = RESULTS_DIR / f"{image_path.stem}_{datetime.now():%Y%m%d_%H%M%S}"
    out_dir.mkdir(parents=True)
    labelled = Image.open(io.BytesIO(base64.b64decode(labelled_b64)))
    labelled.save(out_dir / 'labelled.png')
    df = pd.DataFrame(parsed)
    df.insert(0, 'ID', range(len(df)))
    df.to_csv(out_dir / 'elements.csv', index=False)
    (out_dir / 'source_path.txt').write_text(str(image_path.resolve()) + '\n')

    print(f'{len(df)} elements in {elapsed:.1f}s -> {out_dir}')
    if show:
        plt.figure(figsize=(15, 15)); plt.axis('off'); plt.imshow(labelled); plt.show()
    return out_dir, df

In [ ]:
# Run: paste image path here
IMAGE_PATH = '/mnt/hitchcock/scratch/oberon/OmniParser/imgs/google_page.png'

out_dir, df = run_omniparser(IMAGE_PATH)
df